# Electromagnetism

An interactive notebook covering the fundamental concepts of electromagnetism — from static charges and fields
to electromagnetic waves, induction, optics, and circuits.

## 1 · Coulomb's Law — Electric Force

Two point charges $q_1$ and $q_2$ separated by a distance $r$ exert a force on each other given by **Coulomb's law**:

$$F = k_e \frac{|q_1||q_2|}{r^2}$$

where $k_e \approx 8.99 \times 10^9 \; \text{N m}^2/\text{C}^2$.

* **Like charges** ($q_1 q_2 > 0$) → the force is **repulsive** (directed away from each other).
* **Unlike charges** ($q_1 q_2 < 0$) → the force is **attractive** (directed toward each other).

The force obeys Newton's third law: the force on $q_1$ due to $q_2$ is equal in magnitude and opposite in direction to the force on $q_2$ due to $q_1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

def coulomb_force(q1=10.0, q2=-5.0, x2=5.0):
    k = 8.9875e-1  # scaled for visualisation
    pos1 = np.array([0.0, 5.0])
    pos2 = np.array([x2, 5.0])
    r = np.linalg.norm(pos2 - pos1)
    if r < 0.2:
        r = 0.2
    F = k * abs(q1) * abs(q2) / r**2
    sign = np.sign(q1 * q2)  # +1 repulsion, -1 attraction

    fig, ax = plt.subplots(figsize=(9, 4))
    c1 = 'blue' if q1 > 0 else 'red'
    c2 = 'blue' if q2 > 0 else 'red'
    ax.scatter(*pos1, s=abs(q1)*25, color=c1, zorder=5)
    ax.scatter(*pos2, s=abs(q2)*25, color=c2, zorder=5)
    ax.text(pos1[0], pos1[1], '+' if q1>0 else '−', color='white', ha='center', va='center', fontsize=11, zorder=6)
    ax.text(pos2[0], pos2[1], '+' if q2>0 else '−', color='white', ha='center', va='center', fontsize=11, zorder=6)

    # force arrows
    if sign > 0:  # repulsion
        ax.quiver(pos1[0], pos1[1], -F, 0, angles='xy', scale_units='xy', scale=1, color=c1, width=0.012)
        ax.quiver(pos2[0], pos2[1],  F, 0, angles='xy', scale_units='xy', scale=1, color=c2, width=0.012)
        label = 'Repulsion'
    else:  # attraction
        ax.quiver(pos1[0], pos1[1],  F, 0, angles='xy', scale_units='xy', scale=1, color=c1, width=0.012)
        ax.quiver(pos2[0], pos2[1], -F, 0, angles='xy', scale_units='xy', scale=1, color=c2, width=0.012)
        label = 'Attraction'

    ax.set_xlim(-6, 12)
    ax.set_ylim(2, 8)
    ax.set_aspect('equal')
    ax.set_title(f'Coulomb Force — {label}   |   F = {F:.3f} (scaled)', fontsize=12)
    ax.set_xlabel('x')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(coulomb_force,
         q1=FloatSlider(min=-15, max=15, step=0.5, value=10, description='q₁ (C)'),
         q2=FloatSlider(min=-15, max=15, step=0.5, value=-5, description='q₂ (C)'),
         x2=FloatSlider(min=0.5, max=10, step=0.1, value=5, description='x₂'));

interactive(children=(FloatSlider(value=10.0, description='q₁ (C)', max=15.0, min=-15.0, step=0.5), FloatSlide…

## 2 · Electric Field of a Dipole

A charge $q$ creates an **electric field** in the surrounding space:

$$\vec{E} = k_e \frac{q}{r^2} \hat{r}$$

The field points **away** from positive charges and **toward** negative charges.

For a system of charges, the total field is the vector sum (superposition principle):

$$\vec{E}_{\text{total}} = \sum_i \vec{E}_i$$

Below we visualise the field of a **dipole** — one positive and one negative charge. Move them to see how the field pattern changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def electric_field_dipole(x_pos=0.0, y_pos=0.0, x_neg=3.0, y_neg=-3.0):
    gx = np.linspace(-5, 5, 20)
    gy = np.linspace(-5, 5, 20)
    X, Y = np.meshgrid(gx, gy)
    k = 6.67
    excl = 1.8  # exclusion radius

    def field_from(qx, qy, q):
        dx, dy = X - qx, Y - qy
        r = np.sqrt(dx**2 + dy**2)
        r[r < 0.01] = 0.01
        mag = k * q / r**2
        mag[r < excl] = 0
        return mag * dx / r, mag * dy / r

    Ex_p, Ey_p = field_from(x_pos, y_pos, 1)
    Ex_n, Ey_n = field_from(x_neg, y_neg, -1)

    # mask near both charges
    rp = np.sqrt((X-x_pos)**2 + (Y-y_pos)**2)
    rn = np.sqrt((X-x_neg)**2 + (Y-y_neg)**2)
    mask = (rp < excl) | (rn < excl)
    Ex_p[mask] = 0; Ey_p[mask] = 0
    Ex_n[mask] = 0; Ey_n[mask] = 0

    Ex, Ey = Ex_p + Ex_n, Ey_p + Ey_n

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.quiver(X, Y, Ex, Ey, scale=20, color='purple', alpha=0.7)
    ax.add_patch(plt.Circle((x_pos, y_pos), 0.6, color='blue'))
    ax.add_patch(plt.Circle((x_neg, y_neg), 0.6, color='red'))
    ax.text(x_pos, y_pos, '+', color='white', fontsize=14, ha='center', va='center')
    ax.text(x_neg, y_neg, '−', color='white', fontsize=14, ha='center', va='center')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.set_title('Electric field of a dipole', fontsize=13)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

interact(electric_field_dipole,
         x_pos=FloatSlider(min=-4, max=4, step=0.2, value=0, description='x⁺'),
         y_pos=FloatSlider(min=-4, max=4, step=0.2, value=0, description='y⁺'),
         x_neg=FloatSlider(min=-4, max=4, step=0.2, value=3, description='x⁻'),
         y_neg=FloatSlider(min=-4, max=4, step=0.2, value=-3, description='y⁻'));

interactive(children=(FloatSlider(value=0.0, description='x⁺', max=4.0, min=-4.0, step=0.2), FloatSlider(value…

## 3 · Magnetic Field of a Current Loop (Biot–Savart Law)

A current $I$ flowing through a wire produces a magnetic field.  For an infinitesimal wire element $d\vec{l}$,
the contribution to the field at a point $\vec{r}$ away is

$$d\vec{B} = \frac{\mu_0}{4\pi} \frac{I\, d\vec{l} \times \hat{r}}{r^2}$$

where $\mu_0 = 4\pi\times10^{-7}\;\text{T m/A}$.

For a **circular loop** of radius $R$ carrying current $I$, the field at the centre points along the loop axis
with magnitude $B = \mu_0 I / (2R)$.

The visualisation below numerically integrates the Biot–Savart law over a circular loop and shows the resulting 3-D field.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

def magnetic_field_loop(I=1.0, R=1.0, elev=25, azim=45):
    mu_0 = 4 * np.pi * 1e-7
    theta = np.linspace(0, 2*np.pi, 100)
    dtheta = theta[1] - theta[0]

    gx = np.linspace(-2, 2, 8)
    gy = np.linspace(-2, 2, 8)
    gz = np.linspace(-1, 1, 5)
    X, Y, Z = np.meshgrid(gx, gy, gz)
    Bx = np.zeros_like(X); By = np.zeros_like(Y); Bz = np.zeros_like(Z)

    for th in theta:
        rx, ry, rz = X - R*np.cos(th), Y - R*np.sin(th), Z
        r3 = (rx**2 + ry**2 + rz**2)**1.5
        r3[r3 < 1e-12] = 1e-12
        dlx = -R * np.sin(th) * dtheta
        dly =  R * np.cos(th) * dtheta
        Bx += mu_0/(4*np.pi) * I * (dly*rz) / r3
        By += mu_0/(4*np.pi) * I * (-dlx*rz) / r3
        Bz += mu_0/(4*np.pi) * I * (dlx*ry - dly*rx) / r3

    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(R*np.cos(theta), R*np.sin(theta), np.zeros_like(theta),
            color='royalblue', linewidth=3, label=f'Loop (I={I:.1f} A, R={R:.1f} m)')
    ax.quiver(X, Y, Z, Bx, By, Bz, length=0.5, normalize=True, color='green', alpha=0.5, label='B field')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title('Magnetic field — Biot–Savart', fontsize=13)
    ax.view_init(elev=elev, azim=azim)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

interact(magnetic_field_loop,
         I=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='I (A)'),
         R=FloatSlider(min=0.3, max=2, step=0.1, value=1, description='R (m)'),
         elev=IntSlider(min=0, max=90, step=5, value=25, description='Elevation'),
         azim=IntSlider(min=-180, max=180, step=5, value=45, description='Azimuth'));

interactive(children=(FloatSlider(value=1.0, description='I (A)', max=5.0, min=0.1), FloatSlider(value=1.0, de…

## 4 · Lorentz Force

A charged particle moving with velocity $\vec{v}$ in a magnetic field $\vec{B}$ experiences a force

$$\vec{F} = q\,(\vec{v} \times \vec{B})$$

Key properties:

* The force is always **perpendicular** to both $\vec{v}$ and $\vec{B}$.
* It does **no work** on the particle (it changes direction, not speed).
* Its magnitude is $F = qvB\sin\theta$, where $\theta$ is the angle between $\vec{v}$ and $\vec{B}$.

Below, a uniform field $\vec{B} = B\,\hat{x}$ fills the space.  Adjust the velocity vector and observe the resulting Lorentz force.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def lorentz_force(vx=1.0, vy=0.0, vz=0.0, Bx=1.0):
    q = 1.0
    v = np.array([vx, vy, vz])
    B = np.array([Bx, 0.0, 0.0])
    F = q * np.cross(v, B)

    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')

    # Background B field
    pts = np.linspace(-3, 3, 4)
    Yg, Zg = np.meshgrid(pts, pts)
    Xg = np.zeros_like(Yg)
    ax.quiver(Xg, Yg, Zg, np.ones_like(Xg)*Bx*0.5, 0, 0,
              color='green', alpha=0.3, length=0.8, normalize=True)

    o = [0, 0, 0]
    ax.quiver(*o, *v, color='blue', linewidth=3, arrow_length_ratio=0.15, label=f'v = ({vx:.1f}, {vy:.1f}, {vz:.1f})')
    ax.quiver(*o, *F, color='red',  linewidth=3, arrow_length_ratio=0.15, label=f'F = ({F[0]:.2f}, {F[1]:.2f}, {F[2]:.2f})')

    lim = max(3, np.max(np.abs(np.concatenate([v, F]))))*1.2
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(f'Lorentz Force   |F| = {np.linalg.norm(F):.3f}', fontsize=13)
    ax.set_box_aspect([1,1,1])
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

interact(lorentz_force,
         vx=FloatSlider(min=-5, max=5, step=0.1, value=1, description='vₓ'),
         vy=FloatSlider(min=-5, max=5, step=0.1, value=0, description='vᵧ'),
         vz=FloatSlider(min=-5, max=5, step=0.1, value=0, description='v_z'),
         Bx=FloatSlider(min=-5, max=5, step=0.1, value=1, description='Bₓ'));

interactive(children=(FloatSlider(value=1.0, description='vₓ', max=5.0, min=-5.0), FloatSlider(value=0.0, desc…

## 5 · Electromagnetic Wave

An electromagnetic (EM) wave consists of oscillating electric and magnetic fields that are
**perpendicular to each other** and to the direction of propagation:

$$\vec{E}(y,t) = E_0 \sin(ky - \omega t)\,\hat{z}, \qquad
  \vec{B}(y,t) = B_0 \sin(ky - \omega t)\,\hat{x}$$

The wave travels at $c = \omega/k = 1/\sqrt{\mu_0 \varepsilon_0} \approx 3\times10^8\;\text{m/s}$
in vacuum, with $E_0 = c\,B_0$.

The energy flux is given by the **Poynting vector** $\vec{S} = \frac{1}{\mu_0} \vec{E}\times\vec{B}$,
directed along the propagation axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def em_wave(freq=1.0, phase=0.0, elev=20, azim=-60):
    t = np.linspace(0, 4*np.pi, 200)
    k = freq
    E = np.sin(k * t + phase)
    B = np.sin(k * t + phase)

    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')

    # E-field in xz plane
    ax.plot(np.zeros_like(t), t, E, color='red', linewidth=2, label='E field')
    for i in range(0, len(t), 8):
        ax.quiver(0, t[i], 0, 0, 0, E[i], color='red', alpha=0.4, arrow_length_ratio=0.1)

    # B-field in xy plane
    ax.plot(B, t, np.zeros_like(t), color='green', linewidth=2, label='B field')
    for i in range(0, len(t), 8):
        ax.quiver(0, t[i], 0, B[i], 0, 0, color='green', alpha=0.4, arrow_length_ratio=0.1)

    ax.set_xlabel('X (B)'); ax.set_ylabel('Y (propagation)'); ax.set_zlabel('Z (E)')
    ax.set_title('Electromagnetic wave', fontsize=13)
    ax.view_init(elev=elev, azim=azim)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

interact(em_wave,
         freq=FloatSlider(min=0.3, max=3, step=0.1, value=1, description='Frequency'),
         phase=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0, description='Phase'),
         elev=FloatSlider(min=0, max=90, step=5, value=20, description='Elevation'),
         azim=FloatSlider(min=-180, max=180, step=5, value=-60, description='Azimuth'));

interactive(children=(FloatSlider(value=1.0, description='Frequency', max=3.0, min=0.3), FloatSlider(value=0.0…

## 6 · Anatomy of a Wave

A sinusoidal wave is described by

$$y(x,t) = A \sin(kx + \phi)$$

| Parameter | Symbol | Definition |
|-----------|--------|------------|
| Amplitude | $A$ | Maximum displacement from equilibrium |
| Frequency | $f$ | Number of oscillations per second ($k = 2\pi f$ in angular form) |
| Phase | $\phi$ | Horizontal shift of the waveform |
| Wavelength | $\lambda$ | Spatial period — distance between consecutive peaks, $\lambda = 2\pi / k$ |

Adjust the sliders to see how each parameter reshapes the wave.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def wave_params(A=1.0, f=1.0, phi=0.0):
    x = np.linspace(0, 2*np.pi, 1000)
    y = A * np.sin(f * x + phi)
    wavelength = 2 * np.pi / f if f > 0 else np.inf

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(x, y, 'b', linewidth=2)

    # mark first two peaks for wavelength
    peak1_x = (np.pi/2 - phi) / f
    peak1_x = peak1_x % (2*np.pi / f) if f > 0 else 0
    peak2_x = peak1_x + wavelength
    if 0 <= peak1_x <= x[-1] and 0 <= peak2_x <= x[-1]:
        ax.plot([peak1_x, peak2_x], [A, A], 'r--', linewidth=1.5, label=f'λ = {wavelength:.2f}')
    # amplitude line
    if 0 <= peak1_x <= x[-1]:
        ax.plot([peak1_x, peak1_x], [0, A], 'g--', linewidth=1.5, label=f'A = {A:.2f}')

    ax.axhline(0, color='k', linewidth=0.5, alpha=0.5)
    ax.set_ylim(-2.2, 2.2)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title('Sine wave', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(wave_params,
         A=FloatSlider(min=0.1, max=2, step=0.05, value=1, description='Amplitude'),
         f=FloatSlider(min=0.1, max=10, step=0.1, value=1, description='Frequency'),
         phi=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0, description='Phase'));

interactive(children=(FloatSlider(value=1.0, description='Amplitude', max=2.0, min=0.1, step=0.05), FloatSlide…

## 7 · Reflection

When a ray of light strikes a flat surface, the **law of reflection** states:

$$\theta_i = \theta_r$$

The angle of incidence $\theta_i$ (measured from the **normal** to the surface) equals the angle of
reflection $\theta_r$.  Both the incident ray, the reflected ray, and the normal lie in the same plane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from ipywidgets import interact, FloatSlider

def reflection(angle_deg=45.0):
    a = np.deg2rad(angle_deg)
    fig, ax = plt.subplots(figsize=(7, 5))

    # surface and normal
    ax.axhline(0, color='gray', linewidth=2)
    ax.plot([0, 0], [-0.3, 1.2], color='gray', linestyle='--', alpha=0.5)

    # incident ray
    h = np.tan(np.pi/2 - a)
    ax.annotate('', xy=(0, 0), xytext=(-1, h),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    # reflected ray
    ax.annotate('', xy=(1, h), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))

    # arcs
    ax.add_patch(Arc((0,0), 0.5, 0.5, angle=0, theta1=90, theta2=90+angle_deg, color='blue', lw=1.5))
    ax.add_patch(Arc((0,0), 0.5, 0.5, angle=0, theta1=90-angle_deg, theta2=90, color='red', lw=1.5))

    ax.text(-0.55, 0.55, f'θᵢ={angle_deg:.0f}°', color='blue', fontsize=11)
    ax.text( 0.15, 0.55, f'θᵣ={angle_deg:.0f}°', color='red', fontsize=11)

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-0.4, 1.4)
    ax.set_aspect('equal')
    ax.set_title('Law of Reflection', fontsize=13)
    ax.legend(['Surface', 'Normal'], fontsize=9, loc='upper right')
    plt.tight_layout()
    plt.show()

interact(reflection,
         angle_deg=FloatSlider(min=1, max=89, step=1, value=45, description='θᵢ (°)'));

interactive(children=(FloatSlider(value=45.0, description='θᵢ (°)', max=89.0, min=1.0, step=1.0), Output()), _…

## 8 · Refraction — Snell's Law

When light passes from a medium with refractive index $n_1$ into a medium with index $n_2$,
its direction changes according to **Snell's law**:

$$n_1 \sin\theta_1 = n_2 \sin\theta_2$$

If $n_2 > n_1$ the refracted ray bends **toward** the normal; if $n_2 < n_1$ it bends **away**.

When $\theta_1$ exceeds the **critical angle** $\theta_c = \arcsin(n_2/n_1)$ (only when $n_1>n_2$),
**total internal reflection** occurs and no light enters the second medium.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from ipywidgets import interact, FloatSlider

def refraction(angle_deg=45.0, n2=1.333):
    n1 = 1.0
    a1 = np.deg2rad(angle_deg)
    sin_a2 = (n1 / n2) * np.sin(a1)
    total_reflect = abs(sin_a2) > 1.0

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axhline(0, color='gray', linewidth=2)
    ax.add_patch(plt.Rectangle((-1.5, -1.2), 3, 1.2, color='lightblue', alpha=0.4))
    ax.plot([0, 0], [-1, 1.2], color='gray', linestyle='--', alpha=0.5)

    h_i = np.tan(np.pi/2 - a1)
    ax.annotate('', xy=(0, 0), xytext=(-1, h_i),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))

    if not total_reflect:
        a2 = np.arcsin(sin_a2)
        h_r = np.tan(np.pi/2 - a2)
        ax.annotate('', xy=(1, -h_r), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2))
        ax.add_patch(Arc((0,0), 0.5, 0.5, angle=0, theta1=270, theta2=270+np.degrees(a2), color='green', lw=1.5))
        ax.text(0.15, -0.5, f'θ₂={np.degrees(a2):.1f}°', color='green', fontsize=11)
    else:
        ax.annotate('', xy=(1, h_i), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='--'))
        ax.text(0.3, 0.7, 'Total reflection', color='red', fontsize=11)

    ax.add_patch(Arc((0,0), 0.5, 0.5, angle=0, theta1=90, theta2=90+angle_deg, color='blue', lw=1.5))
    ax.text(-0.6, 0.5, f'θ₁={angle_deg:.0f}°', color='blue', fontsize=11)
    ax.text(-1.4, 0.9, f'n₁ = {n1}', fontsize=10)
    ax.text(-1.4, -0.9, f'n₂ = {n2:.3f}', fontsize=10)

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.2, 1.4)
    ax.set_aspect('equal')
    ax.set_title("Snell's Law", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(refraction,
         angle_deg=FloatSlider(min=1, max=89, step=1, value=45, description='θ₁ (°)'),
         n2=FloatSlider(min=0.5, max=2.5, step=0.01, value=1.333, description='n₂'));

interactive(children=(FloatSlider(value=45.0, description='θ₁ (°)', max=89.0, min=1.0, step=1.0), FloatSlider(…

## 9 · Faraday–Lenz Law

A changing magnetic flux through a conducting loop induces an electromotive force (EMF):

$$\varepsilon = -\frac{d\Phi_B}{dt}, \qquad \Phi_B = \int \vec{B}\cdot d\vec{A}$$

The minus sign (**Lenz's law**) means the induced current flows in a direction that **opposes**
the change in flux that produced it.

Below, a circular coil slides along the $x$-axis through a region of uniform upward magnetic field.
Watch the induced current reverse when the coil enters vs.\ leaves the field region.

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from ipywidgets import interact, FloatSlider

def faraday_lenz(position=-0.8):
    B0 = 1.0
    fx0, fx1 = -0.2, 0.5
    fy0, fy1 = -0.5, 0.5
    R = 0.25

    ov_l = max(position - R, fx0)
    ov_r = min(position + R, fx1)
    ov_frac = max(0, ov_r - ov_l) / (2*R)
    ov_frac = min(ov_frac, 1.0)
    flux = B0 * np.pi * R**2 * ov_frac

    dt = 0.01; v = 0.5
    p2 = position + dt*v
    ov_l2 = max(p2 - R, fx0); ov_r2 = min(p2 + R, fx1)
    ov_frac2 = min(1, max(0, ov_r2 - ov_l2)/(2*R))
    flux2 = B0 * np.pi * R**2 * ov_frac2
    dflux = (flux2 - flux) / dt
    emf = -dflux

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')

    # B-field box
    verts = [
        [[fx0,fy0,-0.5],[fx1,fy0,-0.5],[fx1,fy1,-0.5],[fx0,fy1,-0.5]],
        [[fx0,fy0, 0.5],[fx1,fy0, 0.5],[fx1,fy1, 0.5],[fx0,fy1, 0.5]],
    ]
    ax.add_collection3d(Poly3DCollection(verts, alpha=0.1, facecolor='cyan', edgecolor='blue'))
    for bx in np.linspace(fx0+0.1, fx1-0.1, 3):
        for by in np.linspace(fy0+0.1, fy1-0.1, 3):
            ax.quiver(bx, by, -0.2, 0, 0, 0.35, color='blue', alpha=0.5)

    # coil
    th = np.linspace(0, 2*np.pi, 200)
    color = 'gray'
    if abs(dflux) > 0.01:
        color = 'orange' if dflux > 0 else 'limegreen'
    ax.plot(position + R*np.cos(th), R*np.sin(th), np.zeros(200), color=color, linewidth=4)

    info = f'Φ = {flux:.3f} Wb\ndΦ/dt = {dflux:.3f}\nε = {emf:.3f} V'
    ax.text2D(0.02, 0.92, info, transform=ax.transAxes, fontsize=10, va='top',
              bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    ax.set_xlim(-0.8, 0.8); ax.set_ylim(-0.7, 0.7); ax.set_zlim(-0.6, 0.6)
    ax.set_xlabel('X (motion →)'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title('Faraday–Lenz Law: Induced EMF', fontsize=13, fontweight='bold')
    ax.view_init(elev=25, azim=-70)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(faraday_lenz,
         position=FloatSlider(min=-0.8, max=0.8, step=0.01, value=-0.8, description='Coil x'));

interactive(children=(FloatSlider(value=-0.8, description='Coil x', max=0.8, min=-0.8, step=0.01), Output()), …

## 10 · AC Circuit (RLC)

An **alternating-current** (AC) circuit driven by a sinusoidal voltage source
$V(t)=V_0\sin(\omega t)$ produces a current that also oscillates sinusoidally.

In a series RLC circuit:

| Component | Symbol | Impedance | Phase of current vs. voltage |
|-----------|--------|-----------|-----------------------------|
| Resistor  | $R$    | $R$       | In phase (0°) |
| Capacitor | $C$    | $1/(\omega C)$ | Leads by 90° |
| Inductor  | $L$    | $\omega L$     | Lags by 90° |

**Resonance** occurs when $\omega_0 = 1/\sqrt{LC}$, minimising total impedance.

Below we plot the instantaneous current $I(t) = I_0 \sin(\omega t)$ and the voltage drops
across each component.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def ac_circuit(freq=1.0, R=1.0, L=0.5, C=0.5):
    omega = 2 * np.pi * freq
    XL = omega * L
    XC = 1 / (omega * C) if omega * C > 1e-9 else 1e9
    Z = np.sqrt(R**2 + (XL - XC)**2)
    phi = np.arctan2(XL - XC, R)
    V0 = 1.0
    I0 = V0 / Z

    t = np.linspace(0, 4 / freq, 1000)
    V = V0 * np.sin(omega * t)
    I = I0 * np.sin(omega * t - phi)
    VR = I * R
    VL = I0 * XL * np.sin(omega * t - phi + np.pi/2)
    VC = I0 * XC * np.sin(omega * t - phi - np.pi/2)

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(t, V, 'k-', linewidth=2, label=f'V(t)  [V₀={V0:.1f}]')
    axes[0].plot(t, I, 'b-', linewidth=2, label=f'I(t)  [I₀={I0:.3f}]')
    axes[0].set_ylabel('V / I'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
    axes[0].set_title(f'AC RLC Circuit   Z={Z:.2f} Ω   φ={np.degrees(phi):.1f}°', fontsize=12)

    axes[1].plot(t, VR, 'r', label='V_R')
    axes[1].plot(t, VL, 'purple', label='V_L')
    axes[1].plot(t, VC, 'green', label='V_C')
    axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Voltage')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(ac_circuit,
         freq=FloatSlider(min=0.2, max=5, step=0.1, value=1, description='f (Hz)'),
         R=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='R (Ω)'),
         L=FloatSlider(min=0.01, max=2, step=0.01, value=0.5, description='L (H)'),
         C=FloatSlider(min=0.01, max=2, step=0.01, value=0.5, description='C (F)'));

interactive(children=(FloatSlider(value=1.0, description='f (Hz)', max=5.0, min=0.2), FloatSlider(value=1.0, d…

## 11 · RLC Components — Individual Behaviour

Applying a sinusoidal voltage $V(t) = V_0 \sin(\omega t)$ to each ideal component in isolation:

**Resistor ($R$)** — current and voltage are in phase:
$\;I_R = \dfrac{V_0}{R}\sin(\omega t)$

**Inductor ($L$)** — current **lags** voltage by 90°:
$\;I_L = \dfrac{V_0}{\omega L}\sin\!\left(\omega t - \dfrac{\pi}{2}\right)$

**Capacitor ($C$)** — current **leads** voltage by 90°:
$\;I_C = V_0\,\omega C\,\sin\!\left(\omega t + \dfrac{\pi}{2}\right)$

Understanding these phase relationships is key to analysing any AC circuit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def rlc_components(freq=1.0, R=1.0, L=0.3, C=0.3):
    omega = 2 * np.pi * freq
    V0 = 1.0
    t = np.linspace(0, 3 / freq, 800)
    V = V0 * np.sin(omega * t)

    IR = (V0 / R) * np.sin(omega * t)
    XL = omega * L
    IL = (V0 / XL) * np.sin(omega * t - np.pi/2)
    XC = 1 / (omega * C)
    IC = (V0 / XC) * np.sin(omega * t + np.pi/2) if omega * C > 1e-9 else np.zeros_like(t)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

    for ax, I, label, color, X, phase in [
        (axes[0], IR, 'Resistor', 'red',     R,  '0°'),
        (axes[1], IL, 'Inductor', 'purple',  XL, '−90°'),
        (axes[2], IC, 'Capacitor','green',   XC, '+90°'),
    ]:
        ax.plot(t, V, 'k', linewidth=1.5, label='V(t)')
        ax.plot(t, I, color=color, linewidth=2, label=f'I(t)  φ={phase}')
        ax.set_xlabel('Time (s)')
        ax.set_title(f'{label}  |Z| = {X:.2f} Ω', fontsize=11)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    axes[0].set_ylabel('V / I')
    plt.tight_layout()
    plt.show()

interact(rlc_components,
         freq=FloatSlider(min=0.2, max=5, step=0.1, value=1, description='f (Hz)'),
         R=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='R (Ω)'),
         L=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='L (H)'),
         C=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='C (F)'));

interactive(children=(FloatSlider(value=1.0, description='f (Hz)', max=5.0, min=0.2), FloatSlider(value=1.0, d…

## 12 · Series RLC — Impedance and Resonance

The total impedance of a series RLC circuit is

$$Z = \sqrt{R^2 + (X_L - X_C)^2}, \qquad X_L = \omega L,\; X_C = \frac{1}{\omega C}$$

The current amplitude is $I_0 = V_0 / Z$ and it lags or leads the voltage by

$$\phi = \arctan\frac{X_L - X_C}{R}$$

At **resonance** $\omega_0 = 1/\sqrt{LC}$, the reactances cancel ($X_L = X_C$), so $Z = R$ (minimum)
and $I_0$ is maximum.  The **quality factor** $Q = \omega_0 L / R$ controls the sharpness of the
resonance peak and the bandwidth $\Delta\omega \approx \omega_0 / Q$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def rlc_impedance(R=1.0, L=0.3, C=0.3):
    f0 = 1 / (2 * np.pi * np.sqrt(L * C))
    Q = 2 * np.pi * f0 * L / R

    freqs = np.linspace(0.01, 5 * f0, 1000)
    omega = 2 * np.pi * freqs
    XL = omega * L
    XC = 1 / (omega * C)
    Z = np.sqrt(R**2 + (XL - XC)**2)
    phi = np.arctan2(XL - XC, R)
    I0 = 1.0 / Z  # V0 = 1

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(freqs, Z, 'b', linewidth=2)
    axes[0].axvline(f0, color='red', linestyle='--', alpha=0.6, label=f'f₀ = {f0:.2f} Hz')
    axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('|Z| (Ω)')
    axes[0].set_title('Impedance', fontsize=11)
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    axes[1].plot(freqs, I0, 'darkorange', linewidth=2)
    axes[1].axvline(f0, color='red', linestyle='--', alpha=0.6, label=f'Q = {Q:.2f}')
    axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('I₀ (A)')
    axes[1].set_title('Current amplitude (resonance curve)', fontsize=11)
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    axes[2].plot(freqs, np.degrees(phi), 'green', linewidth=2)
    axes[2].axhline(0, color='gray', linewidth=0.5)
    axes[2].axvline(f0, color='red', linestyle='--', alpha=0.6, label=f'f₀ = {f0:.2f} Hz')
    axes[2].set_xlabel('Frequency (Hz)'); axes[2].set_ylabel('Phase φ (°)')
    axes[2].set_title('Phase angle', fontsize=11)
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    plt.suptitle(f'Series RLC   R={R}Ω  L={L}H  C={C}F   f₀={f0:.2f} Hz   Q={Q:.2f}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

interact(rlc_impedance,
         R=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='R (Ω)'),
         L=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='L (H)'),
         C=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='C (F)'));

interactive(children=(FloatSlider(value=1.0, description='R (Ω)', max=5.0, min=0.1), FloatSlider(value=0.3, de…

## 13 · RLC Step Response and Rise Time

When a DC voltage step $V_0$ is suddenly applied to a series RLC circuit, the response depends
on the **damping ratio**

$$\zeta = \frac{R}{2}\sqrt{\frac{C}{L}}$$

| Regime | Condition | Behaviour |
|--------|-----------|----------|
| Under-damped | $\zeta < 1$ | Oscillatory ringing that decays to $V_0$ |
| Critically damped | $\zeta = 1$ | Fastest approach to $V_0$ without overshoot |
| Over-damped | $\zeta > 1$ | Slow exponential approach to $V_0$ |

The **rise time** $t_r$ (10 % → 90 % of final value) for the under-damped case is approximately

$$t_r \approx \frac{1.8}{\omega_n}, \qquad \omega_n = \frac{1}{\sqrt{LC}}$$

while the settling time scales as $t_s \sim 4/(\zeta\omega_n)$.

In [15]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def rlc_step(R=1.0, L=0.3, C=0.3):
    V0 = 1.0
    wn = 1 / np.sqrt(L * C)
    zeta = (R / 2) * np.sqrt(C / L)

    # State: [i_L, v_C]  (inductor current, capacitor voltage)
    # L di/dt = V0 - Ri - v_C
    # C dv_C/dt = i
    def ode(t, y):
        i, vc = y
        di = (V0 - R * i - vc) / L
        dvc = i / C
        return [di, dvc]

    t_end = max(15 / wn, 5)
    sol = solve_ivp(ode, [0, t_end], [0.0, 0.0], max_step=t_end/2000, dense_output=True)
    t = np.linspace(0, t_end, 3000)
    y = sol.sol(t)
    i_L = y[0]; v_C = y[1]
    v_R = R * i_L
    v_L = V0 - v_R - v_C

    # rise time 10%-90%
    idx10 = np.searchsorted(v_C, 0.1 * V0)
    idx90 = np.searchsorted(v_C, 0.9 * V0)
    if idx10 < len(t) and idx90 < len(t) and idx90 > idx10:
        t_rise = t[idx90] - t[idx10]
    else:
        t_rise = float('nan')

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    axes[0,0].plot(t, v_C, 'b', linewidth=2)
    axes[0,0].axhline(V0, color='gray', linestyle='--', alpha=0.5)
    axes[0,0].axhline(0.1*V0, color='green', linestyle=':', alpha=0.4)
    axes[0,0].axhline(0.9*V0, color='green', linestyle=':', alpha=0.4)
    if not np.isnan(t_rise):
        axes[0,0].axvspan(t[idx10], t[idx90], alpha=0.1, color='green')
    axes[0,0].set_title(f'Capacitor voltage   tᵣ = {t_rise:.3f} s', fontsize=11)
    axes[0,0].set_ylabel('v_C (V)'); axes[0,0].grid(alpha=0.3)

    axes[0,1].plot(t, i_L, 'darkorange', linewidth=2)
    axes[0,1].set_title('Inductor current', fontsize=11)
    axes[0,1].set_ylabel('i_L (A)'); axes[0,1].grid(alpha=0.3)

    axes[1,0].plot(t, v_R, 'red', linewidth=1.5, label='v_R')
    axes[1,0].plot(t, v_L, 'purple', linewidth=1.5, label='v_L')
    axes[1,0].plot(t, v_C, 'b', linewidth=1.5, alpha=0.5, label='v_C')
    axes[1,0].set_title('Component voltages', fontsize=11)
    axes[1,0].set_xlabel('Time (s)'); axes[1,0].set_ylabel('V')
    axes[1,0].legend(fontsize=8); axes[1,0].grid(alpha=0.3)

    # energy
    E_C = 0.5 * C * v_C**2
    E_L = 0.5 * L * i_L**2
    E_R = np.cumsum(v_R * i_L) * (t[1]-t[0])
    axes[1,1].plot(t, E_C, 'b', label='E_C')
    axes[1,1].plot(t, E_L, 'purple', label='E_L')
    axes[1,1].plot(t, E_R, 'red', linestyle='--', label='E_R (dissipated)')
    axes[1,1].set_title('Energy', fontsize=11)
    axes[1,1].set_xlabel('Time (s)'); axes[1,1].set_ylabel('Energy (J)')
    axes[1,1].legend(fontsize=8); axes[1,1].grid(alpha=0.3)

    regime = 'Under-damped' if zeta < 1 else ('Critically damped' if abs(zeta-1) < 0.05 else 'Over-damped')
    fig.suptitle(f'RLC Step Response   ζ = {zeta:.3f}  ({regime})   ωₙ = {wn:.2f} rad/s', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

interact(rlc_step,
         R=FloatSlider(min=0.05, max=8, step=0.05, value=1, description='R (Ω)'),
         L=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='L (H)'),
         C=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='C (F)'));

interactive(children=(FloatSlider(value=1.0, description='R (Ω)', max=8.0, min=0.05, step=0.05), FloatSlider(v…

## 14 · RLC Phasor Diagram

In AC steady state, voltages and currents are represented as **phasors** — rotating vectors in
the complex plane.

For a series RLC circuit driven at angular frequency $\omega$:

* $\vec{V}_R$ is **in phase** with $\vec{I}$ (along the real axis).
* $\vec{V}_L$ **leads** $\vec{I}$ by 90° (pointing up).
* $\vec{V}_C$ **lags** $\vec{I}$ by 90° (pointing down).
* The source voltage phasor $\vec{V}_s = \vec{V}_R + \vec{V}_L + \vec{V}_C$.

The diagram below shows these phasors and the resulting phase angle $\phi$.

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def phasor_diagram(freq=1.0, R=1.0, L=0.3, C=0.3):
    omega = 2 * np.pi * freq
    V0 = 1.0
    XL = omega * L
    XC = 1 / (omega * C)
    Z = np.sqrt(R**2 + (XL - XC)**2)
    I0 = V0 / Z
    phi = np.arctan2(XL - XC, R)

    VR = I0 * R
    VL = I0 * XL
    VC = I0 * XC

    fig, (ax_ph, ax_wave) = plt.subplots(1, 2, figsize=(13, 5))

    # phasor diagram
    ax_ph.set_aspect('equal')
    kw = dict(angles='xy', scale_units='xy', scale=1, width=0.015)
    ax_ph.quiver(0, 0, VR, 0, color='red', **kw, label=f'V_R = {VR:.3f}')
    ax_ph.quiver(VR, 0, 0, VL, color='purple', **kw, label=f'V_L = {VL:.3f}')
    ax_ph.quiver(VR, 0, 0, -VC, color='green', **kw, label=f'V_C = {VC:.3f}')
    Vs_x = VR; Vs_y = VL - VC
    ax_ph.quiver(0, 0, Vs_x, Vs_y, color='black', **kw, label=f'V_s = {V0:.3f}')

    lim = max(abs(VR), abs(VL), abs(VC), V0) * 1.4
    ax_ph.set_xlim(-lim*0.3, lim); ax_ph.set_ylim(-lim, lim)
    ax_ph.axhline(0, color='gray', linewidth=0.5)
    ax_ph.axvline(0, color='gray', linewidth=0.5)
    ax_ph.set_title(f'Phasor diagram  φ = {np.degrees(phi):.1f}°', fontsize=11)
    ax_ph.legend(fontsize=8, loc='lower right'); ax_ph.grid(alpha=0.3)

    # time-domain
    t = np.linspace(0, 3/freq, 800)
    ax_wave.plot(t, V0*np.sin(omega*t), 'k', linewidth=2, label='V_s')
    ax_wave.plot(t, I0*np.sin(omega*t - phi)*Z, 'b--', alpha=0.4, linewidth=1)
    ax_wave.plot(t, VR*np.sin(omega*t - phi), 'red', linewidth=1.5, label='V_R')
    ax_wave.plot(t, VL*np.cos(omega*t - phi), 'purple', linewidth=1.5, label='V_L')
    ax_wave.plot(t, -VC*np.cos(omega*t - phi), 'green', linewidth=1.5, label='V_C')
    ax_wave.set_xlabel('Time (s)'); ax_wave.set_ylabel('Voltage (V)')
    ax_wave.set_title('Time domain', fontsize=11)
    ax_wave.legend(fontsize=8); ax_wave.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(phasor_diagram,
         freq=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='f (Hz)'),
         R=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='R (Ω)'),
         L=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='L (H)'),
         C=FloatSlider(min=0.01, max=2, step=0.01, value=0.3, description='C (F)'));

interactive(children=(FloatSlider(value=1.0, description='f (Hz)', max=5.0, min=0.1), FloatSlider(value=1.0, d…

## 15 · Dipole Antenna — Radiation Pattern

A **half-wave dipole antenna** is one of the simplest radiating structures. When an AC current
oscillates in the antenna, it generates electromagnetic waves.

The far-field radiation intensity of an ideal short dipole follows

$$U(\theta) \propto \sin^2\theta$$

where $\theta$ is measured from the antenna axis.  The pattern is a torus — maximum radiation
perpendicular to the antenna, zero along its axis.

Below we plot the 2-D polar radiation pattern together with the E-field lines
that an oscillating dipole produces.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def antenna_pattern(freq=1.0, time=0.0):
    fig, (ax_pat, ax_field) = plt.subplots(1, 2, figsize=(12, 5),
                                           subplot_kw={'projection': 'polar'})

    # radiation pattern
    theta = np.linspace(0, 2*np.pi, 360)
    U = np.sin(theta)**2
    ax_pat.plot(theta, U, 'b', linewidth=2)
    ax_pat.fill(theta, U, alpha=0.15, color='blue')
    ax_pat.set_title('Radiation pattern U(θ) ∝ sin²θ', fontsize=11, pad=15)

    # EM wave fronts
    omega = 2 * np.pi * freq
    r_vals = np.linspace(0.3, 2.5, 8)
    for r in r_vals:
        phase = omega * time - 2 * r
        amp = np.sin(phase) * np.sin(theta)**2 / (r**0.5 + 0.3)
        c = 'red' if np.sin(phase) > 0 else 'blue'
        ax_field.plot(theta, np.full_like(theta, r) + amp * 0.15,
                      color=c, alpha=0.6, linewidth=1)

    ax_field.plot([0, 0], [0, 0.25], color='black', linewidth=4)
    ax_field.plot([np.pi, np.pi], [0, 0.25], color='black', linewidth=4)
    ax_field.set_ylim(0, 2.8)
    ax_field.set_title('EM wave fronts', fontsize=11, pad=15)

    plt.tight_layout()
    plt.show()

interact(antenna_pattern,
         freq=FloatSlider(min=0.2, max=3, step=0.1, value=1, description='Freq'),
         time=FloatSlider(min=0, max=4, step=0.05, value=0, description='Time'));

interactive(children=(FloatSlider(value=1.0, description='Freq', max=3.0, min=0.2), FloatSlider(value=0.0, des…

## 16 · Interference

When two coherent waves overlap, they **interfere**. At any point in space the resulting
amplitude is the sum of the individual amplitudes:

$$y = y_1 + y_2 = A_1\sin(kx - \omega t) + A_2\sin(kx - \omega t + \delta)$$

* **Constructive interference** — the waves are in phase ($\delta = 0, 2\pi, \ldots$) and amplitudes add.
* **Destructive interference** — the waves are out of phase ($\delta = \pi, 3\pi, \ldots$) and amplitudes cancel.

For **Young's double-slit experiment**, two point sources separated by $d$ produce an intensity
pattern on a distant screen:

$$I(\theta) = I_0 \cos^2\!\left(\frac{\pi d \sin\theta}{\lambda}\right)$$

Bright fringes appear at angles satisfying $d\sin\theta = m\lambda$ ($m = 0, \pm1, \pm2, \ldots$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def interference(A1=1.0, A2=1.0, delta=0.0, d=3.0, lam=1.0):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # left: two-wave superposition
    x = np.linspace(0, 4*np.pi, 600)
    y1 = A1 * np.sin(x)
    y2 = A2 * np.sin(x + delta)
    axes[0].plot(x, y1, 'b', alpha=0.5, label='Wave 1')
    axes[0].plot(x, y2, 'r', alpha=0.5, label='Wave 2')
    axes[0].plot(x, y1 + y2, 'k', linewidth=2, label='Sum')
    axes[0].axhline(0, color='gray', linewidth=0.5)
    axes[0].set_ylim(-4.5, 4.5)
    axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
    axes[0].set_title(f'Superposition  (δ = {delta:.2f} rad)', fontsize=11)
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    # right: double-slit intensity pattern
    theta = np.linspace(-np.pi/3, np.pi/3, 800)
    arg = np.pi * d * np.sin(theta) / lam
    I = np.cos(arg)**2
    axes[1].plot(np.degrees(theta), I, 'purple', linewidth=2)
    axes[1].fill_between(np.degrees(theta), I, alpha=0.15, color='purple')
    axes[1].set_xlabel('θ (°)'); axes[1].set_ylabel('I / I₀')
    axes[1].set_title(f'Double-slit  d={d:.1f}λ', fontsize=11)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(interference,
         A1=FloatSlider(min=0.1, max=2, step=0.05, value=1, description='A₁'),
         A2=FloatSlider(min=0.1, max=2, step=0.05, value=1, description='A₂'),
         delta=FloatSlider(min=0, max=2*np.pi, step=0.05, value=0, description='δ (rad)'),
         d=FloatSlider(min=0.5, max=10, step=0.1, value=3, description='d / λ'),
         lam=FloatSlider(min=0.3, max=3, step=0.1, value=1, description='λ'));

interactive(children=(FloatSlider(value=1.0, description='A₁', max=2.0, min=0.1, step=0.05), FloatSlider(value…

## 17 · Diffraction

When a wave passes through an aperture of width $a$, it spreads out — this is **diffraction**.

For a **single slit** the intensity on a distant screen is

$$I(\theta) = I_0 \left[\frac{\sin\beta}{\beta}\right]^2, \qquad \beta = \frac{\pi a \sin\theta}{\lambda}$$

Minima occur at $a\sin\theta = m\lambda$ ($m = \pm1, \pm2, \ldots$). The central maximum has
angular half-width $\approx \lambda / a$, so a narrower slit produces a broader pattern.

For a **multi-slit grating** ($N$ slits, spacing $d$), sharp principal maxima appear at
$d\sin\theta = m\lambda$ with intensities modulated by the single-slit envelope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

def diffraction(a=2.0, d=6.0, lam=1.0, N=2):
    theta = np.linspace(-np.pi/4, np.pi/4, 2000)
    beta = np.pi * a * np.sin(theta) / lam
    beta = np.where(np.abs(beta) < 1e-12, 1e-12, beta)
    single = (np.sin(beta) / beta)**2

    # multi-slit
    phi = np.pi * d * np.sin(theta) / lam
    phi = np.where(np.abs(phi) < 1e-12, 1e-12, phi)
    multi = (np.sin(N * phi) / (N * np.sin(phi)))**2
    combined = single * multi

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(np.degrees(theta), single, 'darkorange', linewidth=2)
    axes[0].fill_between(np.degrees(theta), single, alpha=0.15, color='orange')
    axes[0].set_title(f'Single slit  a = {a:.1f}λ', fontsize=11)
    axes[0].set_xlabel('θ (°)'); axes[0].set_ylabel('I / I₀')
    axes[0].grid(alpha=0.3)

    axes[1].plot(np.degrees(theta), combined, 'darkgreen', linewidth=1.5, label=f'{N}-slit pattern')
    axes[1].plot(np.degrees(theta), single, 'orange', linewidth=1, alpha=0.6, linestyle='--', label='Envelope')
    axes[1].fill_between(np.degrees(theta), combined, alpha=0.1, color='green')
    axes[1].set_title(f'{N}-slit grating  d = {d:.1f}λ', fontsize=11)
    axes[1].set_xlabel('θ (°)'); axes[1].set_ylabel('I / I₀')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(diffraction,
         a=FloatSlider(min=0.5, max=8, step=0.1, value=2, description='a / λ'),
         d=FloatSlider(min=1, max=20, step=0.5, value=6, description='d / λ'),
         lam=FloatSlider(min=0.3, max=3, step=0.1, value=1, description='λ'),
         N=IntSlider(min=2, max=12, step=1, value=2, description='N slits'));

interactive(children=(FloatSlider(value=2.0, description='a / λ', max=8.0, min=0.5), FloatSlider(value=6.0, de…